# 00 — Preprocessing

This notebook turns the three raw record files into the event logs the analysis builds on. It runs in **five steps**:

1. **Merge** — the exams of the second programme (`students_additional`) are merged into the main records (`students_anonimized`), and the student attributes
   (`students_attributes`) are joined as case attributes;
2. **Encode** — the elective slots lose their roman numeral, and the exam outcome is folded into the activity token;
3. **Filter** — the students the cohort comparison is defined for: complete data, finished studies, then split into the two cohorts.
4. **Write** — Four event logs are written to `data/preprocessed/`

| file | students |
|---|---|
| `students_combined.csv` | every student in the records, unfiltered |
| `students_filtered.csv` | complete data **and** finished |
| `students_on-time.csv` | the filtered students who graduate on time |
| `students_late.csv` | the filtered students who graduate late |


5. **Summary** — Summary table of all four event logs.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

# --- folders
BASE_DIR = Path.cwd()
RAW_DIR = BASE_DIR / "data" / "raw"
PREPROCESSED_DIR = BASE_DIR / "data" / "preprocessed"
PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# --- input files
IN_ANONIMIZED = (
    RAW_DIR / "students_anonimized_26_08_18.csv"
)  # programme 1353, the main records
IN_ADDITIONAL = (
    RAW_DIR / "students_additional_records_26_08_18.csv"
)  # programme 1363, the same students' other exams
IN_ATTRIBUTES = (
    RAW_DIR / "students_attributes_merged_26_08_06.csv"
)  # the case attributes

# --- output files
OUT_FULL = PREPROCESSED_DIR / "students_full.csv"
OUT_FILTERED = PREPROCESSED_DIR / "students_filtered.csv"
OUT_ON_TIME = PREPROCESSED_DIR / "students_on-time.csv"
OUT_LATE = PREPROCESSED_DIR / "students_late.csv"

# --- examinations to drop
# `CNV` ("convalidado") is not a grade but a validation: the exam was sat at a
# different university and only recognised here, so it says nothing about how
# the student moved through THIS programme
TRANSFERRED_GRADE = "CNV"

# --- case attribute encoding parameters
N_BINS = 3  # the numeric attributes are cut into
BIN_LABELS = ["low", "medium", "high"]  # this many quantile bins

## 1. Merge the additional records and the case attributes

### 1.1 Read main records

The main records `students_anonymized` (programme 1353) are read first

- **the single `semester == 3` event** (one student only) is moved into semester 2 of the
  same year, as an artifact of the source data.
- **examinations graded `CNV` are dropped.** `CNV` ("convalidado") is a validation, not a
  grade: the exam was sat at a different university and only recognised here, so it carries
  no behaviour within this programme. 
- **the elective slots lose their roman numeral.** `Elective I` / `II` / `III` all become the
  single course `Elective`. 

In [3]:
raw = pd.read_csv(IN_ANONIMIZED)

# examinations transferred from another university carry no behaviour of their own
n_raw = len(raw)
raw = raw[raw["final_grade"].astype(str) != TRANSFERRED_GRADE].reset_index(drop=True)

# the elective slots are one course: `Elective I`, `Elective II`, ... -> `Elective`
is_elective = raw["course"].str.startswith("Elective ")
raw.loc[is_elective, "course"] = "Elective"

# the semester == 3 artifact: moved into semester 2 of the same year.
raw.loc[~raw["semester"].isin([1, 2]), "semester"] = 2

print(
    f"{IN_ANONIMIZED.name}: {len(raw)} examinations, "
    f"{raw['student_ID'].nunique()} students "
)

students_anonimized_26_08_18.csv: 23526 examinations, 872 students 


### 1.2 Read and merge additional records

`students_additional` holds the exams recorded under the second programme (1363).
Their `CNV` examinations are dropped first, exactly as in the main records. They are then
appended as a **left** merge:

- to the **courses the main records also know**, the second programme's exclusive
  courses are discarded. 
- to the **students the main records know**, since an exam of an unknown student has no
  journey to join — and no cohort flags either.

In [4]:
add = pd.read_csv(IN_ADDITIONAL)
n_add = len(add)

# the same transferred examinations, dropped before anything is merged
n_cnv = int((add["final_grade"].astype(str) == TRANSFERRED_GRADE).sum())
add = add[add["final_grade"].astype(str) != TRANSFERRED_GRADE]

# elective encoding
add.loc[add["course"].str.startswith("Elective "), "course"] = "Elective"

# only courses the main records also know
add = add[add["course"].isin(set(raw["course"].unique()))]
# only students the main records know
add = add[add["student_ID"].isin(set(raw["student_ID"]))]

merged = pd.concat([raw, add], ignore_index=True)
merged = merged.sort_values(["student_ID", "year", "semester"]).reset_index(drop=True)

# a student who appears in BOTH files transferred between the programmes
transferred_ids = set(raw["student_ID"]) & set(add["student_ID"])
print(
    f"{IN_ADDITIONAL.name}: {len(add)}/{n_add} events merged "
    f"({n_cnv} {TRANSFERRED_GRADE} examinations dropped)."
)
print(
    f"merged log: {len(merged)} examinations, {merged['student_ID'].nunique()} students"
)

students_additional_records_26_08_18.csv: 4974/5495 events merged (2 CNV examinations dropped).
merged log: 28500 examinations, 872 students


### 1c. The case attributes

`students_attributes` records the students' attributes. 

- a student has **one row per programme**; the row of the programme under study (1353) wins, and among those the latest entry year;
- every attribute comes from **this table**, the cohort flags `finished`, `in_time` and
  `incomplete_data` included (the records carry the same values, but one source keeps them
  unambiguous). Two are calculated from the records: `transferred` — a student who appears in **both** record files (step 1.2) — and
  the transfer-aware correction of `in_time`: where the records state
  `in_time_considering_transfers`, that flag decides the cohort, so a student who transferred
  within the programme is judged on the semesters they actually spent;
- `chilean` and `special_access` are decoded into words; the remaining codes stay codes, to
  preserve the anonymisation;
- the six **numeric** attributes (high-school grades and rank, the four admission-test scores)
  are cut into `N_BINS` quantile bins over all students, so a bin means the same thing in both
  cohorts;
- `region`, `city` and `district` are encoded **separately**, like every other categorical
  attribute: each level keeps its own code and says nothing about the levels above it.

Everything else that is missing is left **empty**.

In [5]:
raw_attrs = pd.read_csv(IN_ATTRIBUTES)
PROGRAMME_COLUMN = "program_number_in_students_attributes_database"

# one row per student: the programme under study (1353) wins, and among its rows
# the latest entry
raw_attrs = (
    raw_attrs.assign(_other=(raw_attrs[PROGRAMME_COLUMN] != 1353).astype(int))
    .sort_values(["_other", "entry_year"], ascending=[True, False])
    .groupby("student_ID", as_index=False)
    .first()
    .drop(columns="_other")
)

# the columns to keep, and their name in the comparison
KEEP = {
    "finished": "finished",
    "in_time": "in_time",
    "incomplete_data": "incomplete_data",
    "entry_year": "entry_year",
    "gender": "gender",
    "chilean": "nationality",
    "preference": "preference",
    "special_access": "special_access",
    "high_school_graduation_year": "high_school_graduation_year",
    "admission_type": "admission_type",
    "high_school_grades_normalized": "high_school_grades",
    "high_school_rank": "high_school_rank",
    "score_in_language_admission_test": "language_score",
    "score_in_math_admission_test": "math_score",
    "score_in_social_sciences_admission_test": "social_sciences_score",
    "score_in_natural_sciences_admission_test": "natural_sciences_score",
    "high_school_plan_type": "high_school_plan_type",
    "high_school_type_program": "high_school_type_program",
    "high_school_type_funding": "high_school_type_funding",
    "region": "region",
    "city": "city",
    "district": "district",
}
attrs = raw_attrs[["student_ID", *KEEP]].rename(columns=KEEP)

# transferred label is generaded from the additional records
attrs["transferred"] = np.where(attrs["student_ID"].isin(transferred_ids), "yes", "no")

# the cohort flag is made transfer-aware: where the records state
# `in_time_considering_transfers`, that flag decides the cohort
transfer_aware = raw.drop_duplicates("student_ID").set_index("student_ID")[
    "in_time_considering_transfers"
]
attrs["in_time"] = (
    attrs["student_ID"].map(transfer_aware).fillna(attrs["in_time"]).astype(bool)
)

# decode the values that have a meaning
CODES = {
    "nationality": {1: "chilean", 2: "foreign"},
    "special_access": {1: "academic excellence", 2: "articulation program"},
}
for col, mapping in CODES.items():
    attrs[col] = attrs[col].map(mapping)

# int categorical attributes, e.g., `preference=3`; the three location levels are
# among them, each encoded on its own like any other attribute
LOCATION = ["region", "city", "district"]
INT_COLUMNS = [
    "entry_year",
    "gender",
    "preference",
    "high_school_graduation_year",
    "admission_type",
    "high_school_plan_type",
    "high_school_type_program",
    "high_school_type_funding",
    *LOCATION,
]
attrs[INT_COLUMNS] = attrs[INT_COLUMNS].astype("Int64")

# numeric attributes: N_BINS quantile bins, cut over ALL students
BINNED = [
    "high_school_grades",
    "high_school_rank",
    "language_score",
    "math_score",
    "social_sciences_score",
    "natural_sciences_score",
]
print(f"{IN_ATTRIBUTES.name}: {len(attrs)} students")
print(f"\n{N_BINS} quantile bins {BIN_LABELS}, cut over all students with a value:")
for col in BINNED:
    binned, edges = pd.qcut(attrs[col], N_BINS, labels=BIN_LABELS, retbins=True)
    attrs[col] = binned.astype(object)
    print(
        f"  {col:24s} borders {' | '.join(f'{e:.3f}' for e in edges)}"
        f"   ({int(attrs[col].notna().sum())} students, "
        f"{int(attrs[col].isna().sum())} without a value)"
    )


# the case attributes of the written logs, in this order
CASE_ATTRIBUTES = [
    "in_time",
    "finished",
    "incomplete_data",
    "transferred",
    "entry_year",
    "gender",
    "nationality",
    "preference",
    "special_access",
    "admission_type",
    "high_school_graduation_year",
    "high_school_grades",
    "high_school_rank",
    "language_score",
    "math_score",
    "social_sciences_score",
    "natural_sciences_score",
    "high_school_plan_type",
    "high_school_type_program",
    "high_school_type_funding",
    *LOCATION,
]
ATTR_FROM_TABLE = [c for c in CASE_ATTRIBUTES if c in attrs.columns]

# ---  join the attributes onto the merged log
# the attribute table is the source for the columns, the records' own copies of them (entry_year, gender) are dropped before the join
full = merged.drop(columns=[c for c in ATTR_FROM_TABLE if c in merged.columns]).merge(
    attrs[["student_ID", *ATTR_FROM_TABLE]], on="student_ID", how="left"
)
print(
    f"\full log: {len(full)} events, {full['student_ID'].nunique()} students, "
    f"{len(CASE_ATTRIBUTES)} case attributes"
)

students_attributes_merged_26_08_06.csv: 872 students

3 quantile bins ['low', 'medium', 'high'], cut over all students with a value:
  high_school_grades       borders 0.500 | 0.760 | 0.810 | 0.950   (503 students, 369 without a value)
  high_school_rank         borders 0.510 | 0.830 | 0.900 | 1.000   (503 students, 369 without a value)
  language_score           borders 0.250 | 0.660 | 0.730 | 0.920   (504 students, 368 without a value)
  math_score               borders 0.220 | 0.710 | 0.770 | 0.980   (504 students, 368 without a value)
  social_sciences_score    borders 0.420 | 0.640 | 0.747 | 0.980   (140 students, 732 without a value)
  natural_sciences_score   borders 0.410 | 0.660 | 0.730 | 0.930   (382 students, 490 without a value)
ull log: 28500 events, 872 students, 23 case attributes


## 2. Encode the activity vocabulary

Two decisions turn a raw exam record into an event:

- **the exam outcome is folded into the token.** `A` / `R` become `"<course> (pass)"` /
  `"<course> (fail)"`. 
- **the semester becomes the timestamp.** `year` and `semester` fold into one **period** label
  `2012/1`.

In [6]:
# the exam outcome becomes part of the activity
outcome = full["situation"].map({"A": "pass", "R": "fail"})
if outcome.isna().any():
    raise ValueError(
        f"unmapped situation code(s): "
        f"{sorted(full.loc[outcome.isna(), 'situation'].unique())}"
    )
full["activity"] = full["course"] + " (" + outcome + ")"

# the semester an exam was sat in becomes the weakly ordered timestamp
full["period"] = (
    full["year"].astype(int).astype(str)
    + "/"
    + full["semester"].astype(int).astype(str)
)

print(
    f"vocabulary: {full['course'].nunique()} courses x pass/fail "
    f"-> {full['activity'].nunique()} activity tokens"
)
print(
    f"periods: {full['period'].min()} - {full['period'].max()} "
    f"({full['period'].nunique()} semesters)"
)

vocabulary: 41 courses x pass/fail -> 81 activity tokens
periods: 2012/1 - 2023/1 (23 semesters)


## 3. Filter

The cohort comparison is defined for the students whose journey is fully recorded and over:

- students marked `incomplete_data` are **dropped** — their journey is missing events, so a
  gap in the trace would not mean the student skipped a semester;
- only `finished` students are **kept** — the *on time* / *late* split is a statement
  about how long a completed journey took, so it does not exist for someone still enrolled.

The split itself then reads the transfer-aware `in_time` from step 1c. The combined log keeps
every student, so the two files together show exactly what the filter costs.

In [7]:
keep = ~full["incomplete_data"] & full["finished"]
filtered = full[keep].reset_index(drop=True)

on_time = filtered[filtered["in_time"]].reset_index(drop=True)
late = filtered[~filtered["in_time"]].reset_index(drop=True)

## 4. Write the event logs

All four logs get the same XES-like shape: the **case** (`case:concept:name`) is the student, the
**activity** (`concept:name`) is the course with its outcome, the **timestamp**
(`time:timestamp`) is the semester the exam was sat in (`2012/1`). Then one
column per case attribute, constant within a case and prefixed **`case:`**.

In [8]:
# the event columns, and — by the same convention — the case attributes, which are
# trace-level and therefore carry the `case:` prefix
XES = {
    "student_ID": "case:concept:name",
    "activity": "concept:name",
    "period": "time:timestamp",
    **{c: f"case:{c}" for c in CASE_ATTRIBUTES},
}


def as_event_log(df):
    """One log in the shared column layout, sorted by case and time."""
    return (
        df.rename(columns=XES)[list(XES.values())]
        .sort_values(["case:concept:name", "time:timestamp"])
        .reset_index(drop=True)
    )


for path, part in [
    (OUT_FULL, full),
    (OUT_FILTERED, filtered),
    (OUT_ON_TIME, on_time),
    (OUT_LATE, late),
]:
    log = as_event_log(part)
    log.to_csv(path, index=False)
    print(f"{path.name:24s} -> {path.parent.name}/")

students_full.csv        -> preprocessed/
students_filtered.csv    -> preprocessed/
students_on-time.csv     -> preprocessed/
students_late.csv        -> preprocessed/


## 5. Summary

In [9]:
elapsed = raw.groupby("student_ID")["elapsed_semesters"].max()


def summarise(part):
    """One row: size, vocabulary, trace variants and the three averages."""
    students = part["student_ID"].nunique()
    # a trace is the sequence of semester blocks, a block the multiset of that
    # semester's activity tokens
    blocks = part.groupby(["student_ID", "period"])["activity"].apply(
        lambda s: tuple(sorted(s))
    )
    semesters = elapsed.reindex(part["student_ID"].unique())
    return {
        "Students": students,
        "Examinations": len(part),
        "Courses": part["course"].nunique(),
        "Event tokens": part["activity"].nunique(),
        "Trace variants": blocks.groupby("student_ID").apply(tuple).nunique(),
        "Exams per student": len(part) / students,
        # only defined over complete joiurneys
        "Study duration": semesters.mean() if semesters.notna().all() else float("nan"),
        "Pass rate": part["situation"].eq("A").mean(),
    }


summary = pd.DataFrame(
    {
        "full log": summarise(full),
        "complete journeys": summarise(filtered),
        "on time": summarise(on_time),
        "late": summarise(late),
    }
).T

display(
    summary.style.format(
        {
            "Students": "{:,.0f}",
            "Examinations": "{:,.0f}",
            "Courses": "{:,.0f}",
            "Event tokens": "{:,.0f}",
            "Trace variants": "{:,.0f}",
            "Exams per student": "{:.2f}",
            "Study duration": "{:.2f}",
            "Pass rate": "{:.2%}",
        },
        na_rep="n.a.",
    )
)

,Students,Examinations,Courses,Event tokens,Trace variants,Exams per student,Study duration,Pass rate
full log,872,"28,500",41,81,767,32.68,n.a.,78.31%
complete journeys,147,"7,733",41,77,145,52.61,11.43,88.22%
on time,27,"1,171",41,63,25,43.37,8.67,95.56%
late,120,"6,562",41,76,120,54.68,12.05,86.91%
